# Jira / Confluence JSON → связанные Markdown

Автономный конвертер для фиксированного JSON-контракта Spark-выгрузки.
Он использует только стандартную библиотеку Python и **не зависит от LLM-Wiki**.

Конвертер сначала читает все JSON, затем строит двусторонние связи:

- Jira Markdown содержит связанные Content ID Confluence;
- Confluence Markdown содержит все связанные ключи Jira из текущего набора JSON.

1. Запустите настройки — появится `jira_json_input`.
2. Положите туда все JSON-файлы выгрузки.
3. Запустите остальные ячейки.
4. Результат появится в `jira_markdown_output`.

Исходные JSON не изменяются.

Поддерживаются старый формат `issues/confluence_body` и новый плоский `issue_key/pages/body_text`. Контуры входят в идентификаторы. Пустые страницы не теряются, их отсутствие текста явно отмечается. `body_text` не разбирается повторно как HTML. Длинные документы не дробятся.


In [ ]:
from pathlib import Path
from html import unescape
from html.parser import HTMLParser
import hashlib
import json
import re


BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "jira_json_input"
OUTPUT_DIR = BASE_DIR / "jira_markdown_output"

# Email обычно не нужен для базы знаний. При необходимости переключите на True.
INCLUDE_EMAILS = False

EXPECTED_SECTIONS = (
    "issue_key",
    "issues",
    "users",
    "customfield",
    "links",
    "remotelink",
    "confluence_body",
    "traceability",
)

INPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Положите JSON-файлы сюда:", INPUT_DIR.resolve())
print("Markdown будет сохранён сюда:", OUTPUT_DIR.resolve())



In [ ]:
def safe_filename(value):
    value = str(value or "document").strip()
    value = re.sub(r"[^0-9A-Za-zА-Яа-яЁё._-]+", "_", value)
    return value.strip("._-") or "document"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with path.open("r", encoding="utf-8-sig") as stream:
        return json.load(stream)


def as_list(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def first(mapping, *names, default=""):
    if not isinstance(mapping, dict):
        return default
    for name in names:
        value = mapping.get(name)
        if value not in (None, "", [], {}):
            return value
    return default


def scalar_text(value):
    if value in (None, "", [], {}):
        return ""
    if isinstance(value, bool):
        return "да" if value else "нет"
    if isinstance(value, (str, int, float)):
        return str(value).strip()
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def jira_markup_to_markdown(value):
    text = scalar_text(value)
    text = re.sub(r"\{color(?::[^}]*)?\}", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[([^\]|]+)\|([^\]]+)\]", r"[\1](\2)", text)
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def export_root(data):
    if not isinstance(data, dict):
        raise ValueError("Верхний уровень JSON должен быть объектом")
    # Некоторые просмотрщики показывают псевдоузел root, а некоторые выгрузки
    # действительно могут оборачивать полезную структуру в ключ root.
    if "issues" not in data and isinstance(data.get("root"), dict):
        return data["root"]
    return data


def contract_warnings(data):
    root = export_root(data)
    warnings = []
    for section in EXPECTED_SECTIONS:
        if section not in root:
            warnings.append(f"В JSON отсутствует секция `{section}`")
    if "issues" in root and not isinstance(root["issues"], (dict, list)):
        warnings.append("Секция `issues` должна быть объектом или массивом")
    for section in EXPECTED_SECTIONS[2:]:
        if section in root and not isinstance(root[section], list):
            warnings.append(f"Секция `{section}` должна быть массивом")
    return warnings


def walk_dicts(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_dicts(child)


def local_name(name):
    return str(name).split("}", 1)[-1].split(":", 1)[-1].lower()


BROKEN_ENTITIES = {
    "nbsp": " ",
    "quot": '"',
    "apos": "'",
    "mdash": "—",
    "ndash": "–",
    "laquo": "«",
    "raquo": "»",
    "amp": "&",
    "lt": "<",
    "gt": ">",
}


def normalize_entities(value):
    text = unescape(str(value or ""))
    # В части Storage Format сущности встречаются уже без начального `&`.
    for name, replacement in BROKEN_ENTITIES.items():
        text = re.sub(rf"(?<![&A-Za-zА-Яа-я0-9_]){name};", replacement, text, flags=re.IGNORECASE)
    return text.replace("\u00a0", " ").replace("\u200b", "")


class StorageNode:
    def __init__(self, tag, attrs=None):
        self.tag = local_name(tag)
        self.attrs = {local_name(key): normalize_entities(value) for key, value in (attrs or [])}
        self.children = []


class ConfluenceStorageParser(HTMLParser):
    """Терпимый разборщик Confluence Storage Format без внешних библиотек."""

    VOID_TAGS = {"br", "hr", "img", "meta", "link", "input"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.root = StorageNode("root")
        self.stack = [self.root]
        self.warnings = []

    def close_open_table_tags(self, new_tag):
        """Восстанавливает границы строк/ячеек в повреждённых таблицах."""
        if new_tag == "tr":
            # Новый <tr> однозначно завершает предыдущую строку и её ячейку.
            for index in range(len(self.stack) - 1, 0, -1):
                if self.stack[index].tag == "table":
                    break
                if self.stack[index].tag == "tr":
                    del self.stack[index:]
                    self.warnings.append("Автоматически закрыта незавершённая строка таблицы")
                    return
        if new_tag in {"td", "th"}:
            # Новая ячейка завершает предыдущую ячейку в той же строке.
            for index in range(len(self.stack) - 1, 0, -1):
                if self.stack[index].tag in {"tr", "table"}:
                    break
                if self.stack[index].tag in {"td", "th"}:
                    del self.stack[index:]
                    self.warnings.append("Автоматически закрыта незавершённая ячейка таблицы")
                    return

    def handle_starttag(self, tag, attrs):
        normalized_tag = local_name(tag)
        self.close_open_table_tags(normalized_tag)
        node = StorageNode(tag, attrs)
        self.stack[-1].children.append(node)
        # HTML-тег <link> пустой, но Confluence-тег <ac:link> содержит детей.
        if str(tag).lower() not in self.VOID_TAGS:
            self.stack.append(node)

    def handle_startendtag(self, tag, attrs):
        self.stack[-1].children.append(StorageNode(tag, attrs))

    def handle_endtag(self, tag):
        expected = local_name(tag)
        for index in range(len(self.stack) - 1, 0, -1):
            if self.stack[index].tag == expected:
                del self.stack[index:]
                return

    def handle_data(self, data):
        if data:
            self.stack[-1].children.append(normalize_entities(data))

    def unknown_decl(self, data):
        # HTMLParser передаёт сюда CDATA из ac:plain-text-body.
        if str(data).startswith("CDATA["):
            self.stack[-1].children.append(str(data)[6:])


def attr_value(node, name, default=""):
    return node.attrs.get(local_name(name), default)


def iter_nodes(node):
    for child in node.children:
        if isinstance(child, StorageNode):
            yield child
            yield from iter_nodes(child)


def descendants(node, tag):
    expected = local_name(tag)
    return [child for child in iter_nodes(node) if child.tag == expected]


def node_plain_text(node):
    parts = []
    for child in node.children:
        if isinstance(child, StorageNode):
            parts.append(node_plain_text(child))
        else:
            parts.append(child)
    return " ".join(normalize_entities("".join(parts)).split())


def clean_markdown(text):
    text = normalize_entities(text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def render_children(node, warnings):
    parts = []
    for child in node.children:
        parts.append(render_element(child, warnings) if isinstance(child, StorageNode) else child)
    return "".join(parts)


def render_table(node, warnings):
    rows = []
    for row in descendants(node, "tr"):
        cells = []
        for cell in row.children:
            if not isinstance(cell, StorageNode) or cell.tag not in {"th", "td"}:
                continue
            value = clean_markdown(render_children(cell, warnings))
            value = re.sub(r"\s*\n\s*", "<br>", value).replace("|", "\\|")
            cells.append(value)
        if cells:
            rows.append(cells)
    if not rows:
        return ""
    width = max(len(row) for row in rows)
    rows = [row + [""] * (width - len(row)) for row in rows]
    result = ["| " + " | ".join(rows[0]) + " |"]
    result.append("| " + " | ".join(["---"] * width) + " |")
    result.extend("| " + " | ".join(row) + " |" for row in rows[1:])
    return "\n" + "\n".join(result) + "\n"


def macro_parameter(element, parameter_name):
    for parameter in descendants(element, "parameter"):
        if attr_value(parameter, "name") == parameter_name:
            return node_plain_text(parameter)
    return ""


def render_macro(element, warnings):
    name = attr_value(element, "name").lower()
    if name == "toc":
        return ""
    if name == "jira":
        key = macro_parameter(element, "key") or macro_parameter(element, "jqlQuery")
        return f"**Jira:** {key}" if key else "**Jira-макрос**"
    rich_bodies = descendants(element, "rich-text-body")
    plain_bodies = descendants(element, "plain-text-body")
    if name == "expand":
        title = macro_parameter(element, "title")
        body = "\n".join(render_children(item, warnings) for item in rich_bodies)
        heading = f"### {title}\n\n" if title else ""
        return "\n\n" + heading + body + "\n\n"
    if name == "code":
        code = "\n".join(node_plain_text(item) for item in plain_bodies)
        return f"\n\n```\n{code.strip()}\n```\n\n"
    body = "\n".join(render_children(item, warnings) for item in rich_bodies)
    if body.strip():
        warnings.append(f"Макрос Confluence `{name or 'unknown'}` извлечён как обычный текст")
        return "\n\n" + body + "\n\n"
    warnings.append(f"Содержимое макроса Confluence `{name or 'unknown'}` не извлечено")
    return ""


def render_element(element, warnings):
    tag = element.tag
    if tag in {"script", "style"}:
        return ""
    if tag == "structured-macro":
        return render_macro(element, warnings)
    if tag == "table":
        return render_table(element, warnings)
    if tag in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        level = int(tag[1])
        return f"\n\n{'#' * level} {clean_markdown(render_children(element, warnings))}\n\n"
    if tag in {"p", "div", "section", "layout", "layout-section", "layout-cell"}:
        return "\n\n" + render_children(element, warnings) + "\n\n"
    if tag == "br":
        return "\n"
    if tag in {"strong", "b"}:
        return "**" + render_children(element, warnings).strip() + "**"
    if tag in {"em", "i"}:
        return "*" + render_children(element, warnings).strip() + "*"
    if tag == "code":
        return "`" + render_children(element, warnings).strip() + "`"
    if tag == "pre":
        return "\n\n```\n" + node_plain_text(element).strip() + "\n```\n\n"
    if tag == "a":
        label = clean_markdown(render_children(element, warnings))
        href = attr_value(element, "href")
        return f"[{label or href}]({href})" if href else label
    if tag == "li":
        return "\n- " + clean_markdown(render_children(element, warnings))
    if tag in {"ul", "ol"}:
        return "\n" + render_children(element, warnings) + "\n"
    if tag == "attachment":
        filename = attr_value(element, "filename", "неизвестное вложение")
        warnings.append(f"Вложение `{filename}` не было передано в JSON")
        return f"[Вложение: {filename}]"
    if tag == "page":
        return attr_value(element, "content-title")
    if tag == "link":
        labels = descendants(element, "plain-text-link-body")
        label = node_plain_text(labels[0]) if labels else ""
        pages = descendants(element, "page")
        urls = descendants(element, "url")
        attachments = descendants(element, "attachment")
        if pages:
            title = attr_value(pages[0], "content-title")
            return label or title
        if urls:
            href = attr_value(urls[0], "value")
            return f"[{label or href}]({href})" if href else label
        if attachments:
            return render_element(attachments[0], warnings)
        return label
    if tag == "image":
        attachment_nodes = descendants(element, "attachment")
        if attachment_nodes:
            return render_element(attachment_nodes[0], warnings)
        return ""
    if tag in {"parameter", "plain-text-body", "rich-text-body"}:
        return render_children(element, warnings)
    return render_children(element, warnings)


def confluence_storage_to_markdown(body):
    parser = ConfluenceStorageParser()
    try:
        source = str(body or "")
        # Некоторые Spark/JSON-выгрузки повторно экранируют часть Storage
        # Format. HTMLParser декодирует &lt;td&gt; как текст, но не разбирает
        # получившийся тег повторно, поэтому декодируем структурные теги заранее.
        encoded_tag = re.compile(
            r"&lt;\s*/?\s*(?:table|thead|tbody|tfoot|tr|td|th|p|div|span|br|"
            r"h[1-6]|ul|ol|li|a|ac:|ri:)",
            flags=re.IGNORECASE,
        )
        for _ in range(2):
            if not encoded_tag.search(source):
                break
            decoded = unescape(source)
            if decoded == source:
                break
            source = decoded
            parser.warnings.append("Повторно экранированный HTML декодирован перед разбором")

        # В Confluence-таблицах встречаются обычные операторы сравнения прямо
        # в тексте, например: `текущая дата < combo.add.current.end`.
        # HTMLParser воспринимает такой `<` как начало повреждённого тега и
        # может проглотить весь остаток таблицы. Настоящие HTML/Confluence-теги
        # начинаются с буквы, `/`, `!` или `?`, поэтому отдельно экранируем
        # только угловые скобки, за которыми идёт пробел, цифра или конец текста.
        sanitized = re.sub(r"<(?=\s|\d|$)", "&lt;", source)
        if sanitized != source:
            source = sanitized
            parser.warnings.append("Операторы `<` в тексте отделены от HTML-разметки")

        # В выгрузке Confluence встречаются невалидные атрибуты таблиц вида
        # `class= highlight-... title= Background colour : Dark teal 85`.
        # Они не нужны для Markdown, но способны сбить HTMLParser и превратить
        # все строки после одной ячейки в сплошной текст. Оставляем сами
        # структурные теги и содержимое, удаляя только их оформление.
        table_tags_cleaned = re.sub(
            r"<\s*(table|thead|tbody|tfoot|tr|td|th|colgroup|col)\b[^>]*>",
            lambda match: f"<{match.group(1).lower()}>",
            source,
            flags=re.IGNORECASE,
        )
        if table_tags_cleaned != source:
            source = table_tags_cleaned
            parser.warnings.append("Некорректные атрибуты таблицы удалены перед разбором")

        parser.feed(source)
        parser.close()
        markdown = clean_markdown(render_children(parser.root, parser.warnings))
    except Exception as error:
        markdown = clean_markdown(node_plain_text(parser.root))
        parser.warnings.append(f"Неполный разбор Storage Format: {type(error).__name__}: {error}")
    return markdown, list(dict.fromkeys(parser.warnings))




In [ ]:
from typing import Any, Sequence

def _render_jira(
    key: str,
    issue: dict[str, Any],
    container: dict[str, Any],
    source_paths: Sequence[str],
    confluence_ids: Sequence[str],
) -> str:
    title = _jira_title(key, issue)
    metadata = [
        ("Ключ", key),
        ("ID", _first(issue, "issue_id", "id")),
        ("Проект", _first(issue, "project_name", "project_key", "project")),
        ("Тип", _first(issue, "issuetype_name", "issue_type", "type")),
        ("Статус", _first(issue, "status_name", "status")),
        ("Приоритет", _first(issue, "priority_name", "priority")),
        ("Решение", _first(issue, "resolution_name", "resolution")),
        ("Создано", _first(issue, "created", "created_at")),
        ("Обновлено", _first(issue, "updated", "updated_at")),
        ("Контур", issue.get("contour")),
        ("Решено", issue.get("resolutiondate")),
        ("Автор", issue.get("reporter")),
        ("Исполнитель", issue.get("assignee")),
        ("Создатель", issue.get("creator")),
    ]
    lines = [f"# {title}", "", "## Метаданные", ""]
    lines.extend(
        f"- **{label}:** {_text(value)}"
        for label, value in metadata
        if _text(value)
    )
    description = _jira_markup(_first(issue, "description"))
    if description:
        lines.extend(["", "## Описание", "", description])
    lines.extend(["", "## Связанные Confluence", ""])
    lines.extend(
        [f"- `{value}`" for value in confluence_ids]
        or ["Связанные страницы не найдены."]
    )
    custom_fields = container.get("customfield", [])
    if isinstance(custom_fields, list) and custom_fields:
        lines.extend(["", "## Пользовательские поля", ""])
        for field in custom_fields:
            if not isinstance(field, dict):
                continue
            name = _text(_first(field, "field_name", "field_id", "name"))
            value = _jira_markup(
                _first(field, "functional_solution", "business_req", "architecture", "textvalue",
                       "string_value", "numbervalue", "datevalue")
            )
            if value:
                lines.extend([f"### {name or 'Поле'}", "", value, ""])
    lines.extend(_jira_extra_sections(container))
    lines.extend(["", "## Оригинальные JSON", ""])
    lines.extend(f"- `{path}`" for path in source_paths)
    return _clean("\n".join(lines)) + "\n"

def _render_confluence(
    content_id: str,
    page: dict[str, Any],
    body: str,
    source_paths: Sequence[str],
    jira_keys: Sequence[str],
    warnings: Sequence[str],
) -> str:
    title = _confluence_title(content_id, page)
    lines = [
        f"# {title}",
        "",
        "## Метаданные",
        "",
        f"- **Content ID:** {content_id}",
    ]
    for label, names in (
        ("Пространство", ("spacename", "spacekey", "space_name")),
        ("Версия", ("version",)),
        ("Создано", ("creationdate", "created", "created_at")),
        ("Обновлено", ("lastmoddate", "updated", "updated_at")),
        ("Контур", ("conf_contour",)),
        ("URL", ("url",)),
        ("Статус выгрузки", ("status",)),
        ("Статус страницы", ("content_status",)),
        ("Текст действителен до", ("body_validto",)),
    ):
        value = _text(_first(page, *names))
        if value:
            lines.append(f"- **{label}:** {value}")
    lines.extend(["", "## Связанные Jira", ""])
    lines.extend([f"- `{key}`" for key in jira_keys] or ["Связанные Jira не найдены."])
    lines.extend(["", "## Содержимое", "", body or "Текст не извлечён."])
    if warnings:
        lines.extend(["", "## Предупреждения извлечения", ""])
        lines.extend(f"- {item}" for item in warnings)
    lines.extend(["", "## Оригинальные JSON", ""])
    lines.extend(f"- `{path}`" for path in source_paths)
    return _clean("\n".join(lines)) + "\n"

def _export_root(value: dict[str, Any]) -> dict[str, Any]:
    if "issues" not in value and isinstance(value.get("root"), dict):
        value = value["root"]
    if "issues" not in value and "issue_key" in value:
        return _normalize_analyst_export(value)
    return value

def _scoped_id(contour: Any, identity: Any) -> str:
    identity = _text(identity)
    return f"{_text(contour)}:{identity}" if contour and identity else identity

def _normalize_analyst_export(value: dict[str, Any]) -> dict[str, Any]:
    """Новая плоская выгрузка → прежний внутренний контракт; оригинал не меняется."""
    contour = _first(value, "contour", "jira_contour")
    children = {"users", "customfields", "links", "remotelinks", "testcases",
                "testresults", "comments", "changelog", "attachments", "pages", "data_quality"}
    issue = {key: item for key, item in value.items() if key not in children}
    issue["issue_key"] = _scoped_id(contour, value.get("issue_key"))
    pages = []
    for page in value.get("pages") or []:
        if not isinstance(page, dict):
            continue
        item = dict(page)
        # Контур Confluence независим от контура Jira. Не угадываем его по Jira.
        scope = _text(page.get("conf_contour"))
        if not scope or scope == "other":
            url = _text(page.get("url"))
            match = re.match(r"https?://([^/]+)", url, re.IGNORECASE)
            scope = match.group(1).lower() if match else "unknown"
        item.update(contentid=_scoped_id(scope, page.get("page_id")),
                    title=page.get("page_title"), version=page.get("page_version"),
                    _format="analyst-v2")
        pages.append(item)
    result = dict(value)
    result.update(issues=issue, issue_key=issue["issue_key"],
                  customfield=value.get("customfields") or [],
                  remotelink=value.get("remotelinks") or [],
                  traceability=value.get("testcases") or [],
                  confluence_body=pages, _format="analyst-v2")
    return result

def _page_text(page: dict[str, Any]) -> tuple[str, list[str]]:
    if "body_text" not in page:
        body, warnings = _confluence_to_markdown(_body_value(page))
    else:
        # Аналитики уже сняли HTML-теги. Повторный HTML-разбор съест <ID>,
        # сравнения и технические обозначения, оставшиеся в обычном тексте.
        body = _text(page.get("body_text"))
        warnings = ["body_text — очищенный текст; структура таблиц, HTML и часть ссылок "
                    "не сохранены выгрузкой и не могут быть восстановлены."]
    status = _text(page.get("status"))
    if not body.strip():
        warnings.append("Текст отсутствует в выгрузке; это не означает, что страница пуста в Confluence.")
    if status and status != "ok":
        warnings.append(f"Статус выгрузки: {status}; полнота и актуальность содержания не подтверждены.")
    if page.get("body_validto"):
        warnings.append(f"Текст из исторической записи, body_validto={page['body_validto']}.")
    if _text(page.get("macro_count")) not in ("", "0"):
        warnings.append("На странице есть макросы; их раскрытое содержимое может отсутствовать.")
    if str(page.get("has_excerpt_include", "")).lower() in ("true", "1"):
        warnings.append("excerpt-include: текст включённой страницы может отсутствовать.")
    return body, warnings

def _jira_extra_sections(container: dict[str, Any]) -> list[str]:
    """Сохраняем дополнительные факты, без загрузки вложений и выполнения кода."""
    lines: list[str] = []
    for name, title in (
        ("users", "Участники"), ("links", "Связанные задачи"),
        ("remotelink", "Внешние документы"), ("comments", "Комментарии"),
        ("changelog", "История изменений"), ("traceability", "Тест-кейсы"),
        ("testresults", "Результаты тестов"), ("attachments", "Вложения (только метаданные)"),
    ):
        records = container.get(name) or []
        if not isinstance(records, list) or not records:
            continue
        lines.extend(["", f"## {title}", ""])
        for i, record in enumerate(records, 1):
            if not isinstance(record, dict):
                continue
            lines.append(f"### Запись {i}")
            for field, value in record.items():
                if field in {"email_address", "email", "emailAddress", "issue_id", "issue_key"}:
                    continue
                text = _jira_markup(value)
                if text:
                    lines.extend([f"- **{field}:** {text}"])
    quality = container.get("data_quality")
    if isinstance(quality, dict) and quality:
        lines.extend(["", "## Качество данных (по выгрузке)", ""])
        lines.extend(f"- **{key}:** {_text(value)}" for key, value in quality.items())
    return lines

def _body_value(page: dict[str, Any]) -> str:
    if "body_text" in page:
        return _text(page.get("body_text"))
    body = page.get("body")
    if isinstance(body, str):
        return body
    if isinstance(body, dict):
        for name in ("value", "storage", "content"):
            value = body.get(name)
            if isinstance(value, str):
                return value
            if isinstance(value, dict) and isinstance(value.get("value"), str):
                return value["value"]
    return ""

def _page_order(item: dict[str, Any]) -> tuple[int, str, bool, str]:
    page = item["page"]
    raw_version = _text(_first(page, "version"))
    try:
        version = int(raw_version)
    except ValueError:
        version = -1
    return (
        version,
        _text(_first(page, "lastmoddate", "updated", "updated_at")),
        bool(_body_value(page).strip()),
        item["source_path"],
    )

def _jira_title(key: str, issue: dict[str, Any]) -> str:
    summary = _text(_first(issue, "summary", "title"))
    return f"{key} — {summary}" if summary else key

def _confluence_title(content_id: str, page: dict[str, Any]) -> str:
    title = _text(_first(page, "title", "name"))
    return f"{title} (Confluence {content_id})" if title else f"Confluence {content_id}"

def _first(mapping: dict[str, Any], *names: str) -> Any:
    for name in names:
        value = mapping.get(name)
        if value not in (None, "", [], {}):
            return value
    return ""

def _text(value: Any) -> str:
    if value in (None, "", [], {}):
        return ""
    if isinstance(value, (str, int, float)):
        return str(value).strip()
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def _jira_markup(value: Any) -> str:
    text = _text(value)
    text = re.sub(r"\{color(?::[^}]*)?\}", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[([^\]|]+)\|([^\]]+)\]", r"[\1](\2)", text)
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()

def _clean(value: str) -> str:
    value = re.sub(r"[ \t]+\n", "\n", value)
    value = re.sub(r"\n{3,}", "\n\n", value)
    return value.strip()

export_root = _export_root
_confluence_to_markdown = confluence_storage_to_markdown

def user_map(container):
    result = {}
    users = first(container, "users", "user", default=[])
    for user in as_list(users):
        if not isinstance(user, dict):
            continue
        keys = {
            scalar_text(first(user, "user_key", "key", "accountId")),
            scalar_text(first(user, "lower_user_name", "username", "name")),
        }
        display = scalar_text(first(user, "display_name", "displayName", "full_name"))
        email = scalar_text(first(user, "email_address", "emailAddress", "email"))
        if INCLUDE_EMAILS and email:
            display = f"{display or next(iter(keys), '')} <{email}>"
        display = display or next((key for key in keys if key), "")
        for key in keys:
            if key:
                result[key] = display
    return result


def resolve_user(value, users):
    if isinstance(value, dict):
        display = scalar_text(first(value, "display_name", "displayName", "full_name"))
        key = scalar_text(first(value, "user_key", "key", "accountId", "name"))
        return display or users.get(key, key)
    key = scalar_text(value)
    return users.get(key, key)


def issue_objects(value):
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    if not isinstance(value, dict):
        return []
    if any(key in value for key in ("summary", "issuenum", "issue_id", "status_name")):
        return [value]
    return [item for item in value.values() if isinstance(item, dict)]


def extract_jira_records(data):
    records = []
    seen = set()
    for container in walk_dicts(data):
        if "issues" not in container:
            continue
        users = user_map(container)
        for issue in issue_objects(container.get("issues")):
            key = scalar_text(first(issue, "issue_key", "key")) or scalar_text(container.get("issue_key"))
            signature = (key, scalar_text(first(issue, "updated", "updated_at")), scalar_text(first(issue, "issue_id", "id")))
            if signature in seen:
                continue
            seen.add(signature)
            records.append((container, issue, users, key))
    if records:
        return records
    for issue in walk_dicts(data):
        if not any(key in issue for key in ("issue_key", "key")):
            continue
        if not any(key in issue for key in ("summary", "description", "status_name")):
            continue
        key = scalar_text(first(issue, "issue_key", "key"))
        signature = (key, scalar_text(first(issue, "updated", "updated_at")), scalar_text(first(issue, "issue_id", "id")))
        if signature not in seen:
            seen.add(signature)
            records.append((issue, issue, user_map(issue), key))
    return records


def collect_links(container, issue):
    values = []
    for owner in (container, issue):
        for name in ("links", "issuelinks", "issue_links"):
            values.extend(as_list(owner.get(name)) if isinstance(owner, dict) else [])
    result = []
    for link in values:
        if not isinstance(link, dict):
            continue
        destination = scalar_text(first(link, "destination_issue_key", "issue_key", "key", "outwardIssue", "inwardIssue"))
        direction = scalar_text(first(link, "direction", "link_direction"))
        relation = scalar_text(first(link, "link_type_name", "type", "relationship"))
        if destination:
            result.append(" — ".join(part for part in (direction, destination, relation) if part))
    return list(dict.fromkeys(result))


def collect_remote_links(container, issue):
    values = []
    for owner in (container, issue):
        for name in ("remotelink", "remoteLink", "remote_links", "remotelinks"):
            values.extend(as_list(owner.get(name)) if isinstance(owner, dict) else [])
    result = []
    for link in values:
        if not isinstance(link, dict):
            continue
        title = scalar_text(first(link, "title", "name"))
        url = scalar_text(first(link, "url", "link", "href"))
        relationship = scalar_text(first(link, "relationship", "relation"))
        if title or url:
            label = title or url
            rendered = f"[{label}]({url})" if url else label
            if relationship:
                rendered += f" — {relationship}"
            result.append(rendered)
    return list(dict.fromkeys(result))


def custom_field_records(container):
    records = first(
        container,
        "customfield",
        "customfields",
        "custom_fields",
        default=[],
    )
    return [item for item in as_list(records) if isinstance(item, dict)]


def render_custom_fields(container):
    records = custom_field_records(container)
    if not records:
        return []
    lines = ["", "## Пользовательские поля Jira", ""]
    for number, record in enumerate(records, 1):
        field_name = scalar_text(first(record, "field_name", "name"))
        field_id = scalar_text(first(record, "field_id", "id"))
        heading = field_name or field_id or f"Поле {number}"
        if field_id and field_id != heading:
            heading += f" (`{field_id}`)"
        lines.extend([f"### {heading}", ""])
        rendered_values = [
            ("Бизнес-требования", first(record, "business_req")),
            ("Функциональное решение", first(record, "functional_solution")),
            ("Архитектура", first(record, "architecture")),
        ]
        has_derived_value = False
        for label, value in rendered_values:
            rendered = jira_markup_to_markdown(value)
            if rendered:
                has_derived_value = True
                lines.extend([f"**{label}:**", "", rendered, ""])
        original = jira_markup_to_markdown(
            first(record, "textvalue", "text_value", "value")
        )
        if original and not has_derived_value:
            lines.extend([original, ""])
        elif original:
            lines.extend(["<details>", "<summary>Исходное значение поля</summary>", "", original, "", "</details>", ""])
    return lines


TRACE_LABELS = (
    ("trace_link_id", "ID связи"),
    ("test_case_id", "ID тест-кейса"),
    ("key", "Ключ тест-кейса"),
    ("latest_version", "Последняя версия"),
    ("archived", "Архивирован"),
    ("name", "Название"),
    ("objective", "Цель"),
    ("precondition", "Предусловия"),
    ("created_on", "Создано"),
    ("updated_on", "Обновлено"),
    ("major_version", "Основная версия"),
    ("test_script_id", "ID тестового скрипта"),
    ("priority_id", "ID приоритета"),
    ("project_id", "ID проекта"),
    ("status_id", "ID статуса"),
    ("created_by", "Создал"),
    ("updated_by", "Обновил"),
    ("owner", "Владелец"),
)


def render_traceability(container):
    records = first(container, "traceability", default=[])
    records = [item for item in as_list(records) if isinstance(item, dict)]
    if not records:
        return []
    lines = ["", "## Трассировка и тест-кейсы", ""]
    for number, record in enumerate(records, 1):
        identity = scalar_text(first(record, "key", "name", "test_case_id"))
        lines.extend([f"### {identity or f'Запись {number}'}", ""])
        for field, label in TRACE_LABELS:
            rendered = scalar_text(record.get(field))
            if rendered:
                lines.append(f"- **{label}:** {rendered}")
        lines.append("")
    return lines


def render_jira_issue(container, issue, users, key, source_name, source_hash, related_confluence_ids=()):
    if container.get("_format") == "analyst-v2":
        text = _render_jira(key, issue, container, [source_name], related_confluence_ids)
        return text + f"\n- SHA-256: `{source_hash}`\n"
    summary = scalar_text(first(issue, "summary", "title"))
    title = " — ".join(part for part in (key, summary) if part) or "Jira issue"
    lines = [f"# {title}", "", "## Метаданные", ""]
    metadata = [
        ("Ключ", key),
        ("ID", first(issue, "issue_id", "id")),
        ("Проект", first(issue, "project_name", "project_key", "project")),
        ("Тип", first(issue, "issuetype_name", "issue_type", "type")),
        ("Статус", first(issue, "status_name", "status")),
        ("Категория статуса", first(issue, "status_category")),
        ("Приоритет", first(issue, "priority_name", "priority")),
        ("Решение", first(issue, "resolution_name", "resolution")),
        ("Создано", first(issue, "created", "created_at")),
        ("Обновлено", first(issue, "updated", "updated_at")),
        ("Решено", first(issue, "resolutiondate", "resolution_date")),
        ("Срок", first(issue, "duedate", "due_date")),
        ("Автор", resolve_user(first(issue, "reporter"), users)),
        ("Исполнитель", resolve_user(first(issue, "assignee"), users)),
        ("Создатель", resolve_user(first(issue, "creator"), users)),
    ]
    for label, value in metadata:
        rendered = scalar_text(value)
        if rendered:
            lines.append(f"- **{label}:** {rendered}")
    description = jira_markup_to_markdown(first(issue, "description"))
    environment = jira_markup_to_markdown(first(issue, "environment"))
    if description:
        lines.extend(["", "## Описание", "", description])
    if environment:
        lines.extend(["", "## Окружение", "", environment])
    links = collect_links(container, issue)
    if links:
        lines.extend(["", "## Связанные задачи", ""])
        lines.extend(f"- {link}" for link in links)
    remote_links = collect_remote_links(container, issue)
    if remote_links:
        lines.extend(["", "## Внешние документы", ""])
        lines.extend(f"- {link}" for link in remote_links)
    lines.extend(render_custom_fields(container))
    lines.extend(render_traceability(container))
    lines.extend(["", "## Связанные Confluence", ""])
    if related_confluence_ids:
        lines.extend(f"- `{content_id}`" for content_id in related_confluence_ids)
    else:
        lines.append("Связанные страницы Confluence не найдены.")
    lines.extend([
        "",
        "## Происхождение",
        "",
        f"- JSON-файл: `{source_name}`",
        f"- SHA-256: `{source_hash}`",
    ])
    return clean_markdown("\n".join(lines)) + "\n"


def body_value(document):
    body = document.get("body") if isinstance(document, dict) else None
    if isinstance(body, str):
        return body
    if isinstance(body, dict):
        for key in ("value", "storage", "content"):
            value = body.get(key)
            if isinstance(value, str):
                return value
            if isinstance(value, dict) and isinstance(value.get("value"), str):
                return value["value"]
    return ""


def extract_confluence_records(data):
    if data.get("_format") == "analyst-v2":
        return data["confluence_body"]
    result = []
    seen = set()
    for document in walk_dicts(data):
        body = body_value(document)
        if not body or "<" not in body:
            continue
        if not any(name in document for name in ("contentid", "content_id", "spacekey", "spacename", "version")):
            continue
        signature = (
            scalar_text(first(document, "contentid", "content_id", "id")),
            scalar_text(first(document, "version")),
            hashlib.sha256(body.encode("utf-8")).hexdigest(),
        )
        if signature not in seen:
            seen.add(signature)
            result.append(document)
    return result


def render_confluence_page(document, source_name, source_hash, related_jira_keys=()):
    if document.get("_format") == "analyst-v2":
        body, warnings = _page_text(document)
        text = _render_confluence(document["contentid"], document, body, [source_name], related_jira_keys, warnings)
        return text + f"\n- SHA-256: `{source_hash}`\n", warnings
    title = scalar_text(first(document, "title", "name")) or "Страница Confluence"
    body = body_value(document)
    markdown, warnings = confluence_storage_to_markdown(body)
    lines = [f"# {title}", "", "## Метаданные", ""]
    common_metadata = []
    metadata = [
        ("Content ID", first(document, "contentid", "content_id", "id")),
        ("Пространство", first(document, "spacename", "spacekey", "space_name")),
        ("Версия", first(document, "version")),
        ("Создано", first(document, "creationdate", "created", "created_at")),
        ("Обновлено", first(document, "lastmoddate", "updated", "updated_at")),
        ("Комментарий версии", first(document, "versioncomment", "version_comment")),
    ]
    for label, value in metadata:
        rendered = scalar_text(value)
        if rendered:
            common_metadata.append(f"- **{label}:** {rendered}")
    lines.extend(common_metadata)
    lines.extend(["", "## Связанные Jira", ""])
    if related_jira_keys:
        lines.extend(f"- `{key}`" for key in related_jira_keys)
    else:
        lines.append("Связанные Jira не найдены.")
    lines.extend(["", "## Содержимое", "", markdown or "Текст страницы не извлечён."])
    if warnings:
        lines.extend(["", "## Предупреждения извлечения", ""])
        lines.extend(f"- {warning}" for warning in warnings)
    lines.extend(["", "## Происхождение", "", f"- JSON-файл: `{source_name}`", f"- SHA-256: `{source_hash}`"])
    return clean_markdown("\n".join(lines)) + "\n", warnings


def version_suffix(value):
    value = scalar_text(value)
    return safe_filename(value.replace(":", "-").replace("+", "_"))[:40]


def convert_all():
    """Два прохода: сначала все связи, затем Markdown."""
    json_files = sorted(INPUT_DIR.glob("*.json"))
    report = {"input_directory": str(INPUT_DIR.resolve()), "files": [], "errors": []}
    if not json_files:
        print("JSON-файлы не найдены. Положите их в", INPUT_DIR.resolve())
        return report

    datasets = []
    jira_to_confluence = {}
    confluence_to_jira = {}
    jira_records = {}
    confluence_records = {}

    # Проход 1: читаем все JSON и строим полную карту связей.
    for path in json_files:
        try:
            data = load_json(path)
            root = export_root(data)
            source_hash = sha256_file(path)
            issues = extract_jira_records(root)
            pages = extract_confluence_records(root)
            issue_keys = {record[3] for record in issues if record[3]}
            content_ids = {
                scalar_text(first(page, "contentid", "content_id", "id"))
                for page in pages
            }
            content_ids.discard("")
            datasets.append((path, root, source_hash, issues, pages))

            for key in issue_keys:
                jira_to_confluence.setdefault(key, set()).update(content_ids)
            for content_id in content_ids:
                confluence_to_jira.setdefault(content_id, set()).update(issue_keys)

            for container, issue, users, key in issues:
                if not key:
                    continue
                order = (
                    scalar_text(first(issue, "updated", "updated_at")),
                    path.name,
                )
                current = jira_records.get(key)
                if current is None or order >= current[0]:
                    jira_records[key] = (order, path, source_hash, container, issue, users)

            for page in pages:
                content_id = scalar_text(first(page, "contentid", "content_id", "id"))
                if not content_id:
                    continue
                raw_version = scalar_text(first(page, "version"))
                try:
                    version = int(raw_version)
                except ValueError:
                    version = -1
                order = (
                    version,
                    scalar_text(first(page, "lastmoddate", "updated", "updated_at")),
                    bool(_body_value(page).strip()),
                    path.name,
                )
                current = confluence_records.get(content_id)
                if current is None or order >= current[0]:
                    confluence_records[content_id] = (order, path, source_hash, page)
        except Exception as error:
            message = f"{path.name}: {type(error).__name__}: {error}"
            report["errors"].append(message)
            print("ОШИБКА:", message)

    # Проход 2: один актуальный Markdown на Jira и Content ID.
    outputs_by_source = {path.name: [] for path in json_files}
    warnings_by_source = {path.name: [] for path in json_files}

    for key, (_, path, source_hash, container, issue, users) in sorted(jira_records.items()):
        filename = f"jira_{safe_filename(key)}.md"
        content = render_jira_issue(
            container,
            issue,
            users,
            key,
            path.name,
            source_hash,
            sorted(jira_to_confluence.get(key, set())),
        )
        target = OUTPUT_DIR / filename
        target.write_text(content, encoding="utf-8")
        outputs_by_source[path.name].append(filename)

    for content_id, (_, path, source_hash, page) in sorted(confluence_records.items()):
        filename = f"confluence_{safe_filename(content_id)}.md"
        content, page_warnings = render_confluence_page(
            page,
            path.name,
            source_hash,
            sorted(confluence_to_jira.get(content_id, set())),
        )
        target = OUTPUT_DIR / filename
        target.write_text(content, encoding="utf-8")
        outputs_by_source[path.name].append(filename)
        warnings_by_source[path.name].extend(
            f"{filename}: {warning}" for warning in page_warnings
        )

    for path, data, source_hash, issues, pages in datasets:
        warnings = contract_warnings(data)
        warnings.extend(warnings_by_source[path.name])
        outputs = outputs_by_source[path.name]
        report["files"].append({
            "source": path.name,
            "outputs": outputs,
            "warnings": warnings,
        })
        print(f"{path.name}: актуальных файлов — {len(outputs)}")
        for filename in outputs:
            print("  ", filename)
        for warning in warnings:
            print("  ПРЕДУПРЕЖДЕНИЕ:", warning)

    report["relations"] = {
        "jira_to_confluence": {
            key: sorted(values) for key, values in sorted(jira_to_confluence.items())
        },
        "confluence_to_jira": {
            key: sorted(values) for key, values in sorted(confluence_to_jira.items())
        },
    }
    report_path = OUTPUT_DIR / "conversion_report.json"
    report_path.write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("\nJira:", len(jira_records))
    print("Confluence:", len(confluence_records))
    print("Отчёт:", report_path.resolve())
    return report


In [ ]:
report = convert_all()



## Просмотр результата

Ячейка ниже показывает начало каждого созданного Markdown-файла.



In [ ]:
for markdown_file in sorted(OUTPUT_DIR.glob("*.md")):
    print("\n" + "=" * 80)
    print(markdown_file.name)
    print("=" * 80)
    lines = markdown_file.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[:120]))
    if len(lines) > 120:
        print(f"\n... ещё строк: {len(lines) - 120}")
